# Toxic Comment Detection — LSTM & TF-IDF MLP

Two models trained on the Jigsaw Toxic Comment dataset:
1. **Bidirectional LSTM** with custom vocabulary
2. **TF-IDF + sklearn MLPClassifier**

Both models follow the same architecture:
- **Phase 1**: Train on 80/20 split → tune threshold on validation set
- **Phase 2**: Retrain on full `train.csv` → evaluate on official `test.csv`

In [ ]:
!pip install torch pandas scikit-learn tqdm imbalanced-learn

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from imblearn.under_sampling import RandomUnderSampler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve
)

## 1. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv("train.csv")

df.head()
print(df.shape)

In [ ]:
toxic_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

df["label"] = df[toxic_cols].max(axis=1)
df["label"].value_counts()

In [ ]:
df = df[["comment_text","label"]]

df.head()

In [ ]:
# Shared preprocessing for both models

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"(.)\1{3,}", r"\1\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return text.strip()

print("Preprocessing training text...")
df["comment_text"] = df["comment_text"].apply(preprocess)
print("Done.")

In [ ]:
# Load official test set (used by BOTH models in Phase 2)

test_df = pd.read_csv("test.csv")
test_labels_df = pd.read_csv("test_labels.csv")

test_df = test_df.merge(
    test_labels_df,
    on="id"
)

test_df = test_df[
    test_df["toxic"] != -1
].copy()

toxic_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

test_df["label"] = (
    test_df[toxic_cols]
    .max(axis=1)
)

test_df["comment_text"] = (
    test_df["comment_text"].apply(preprocess)
)

print(f"Official test set: {test_df.shape}")
print(test_df["label"].value_counts())

In [ ]:
# Same split used by BOTH models for Phase 1

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["comment_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

## 2. LSTM — Phase 1: Train on Split, Evaluate on Validation

In [ ]:
#we tokenize the comments
def tokenize(text):
    return text.lower().split()

In [ ]:
counter = Counter()

for text in train_texts:
    counter.update(tokenize(text))

vocab_size = 20000

most_common = counter.most_common(vocab_size)

word2idx = {
    "<PAD>":0,
    "<UNK>":1
}

for word,_ in most_common:
    word2idx[word] = len(word2idx)

len(word2idx)

In [ ]:
def encode(text):

    tokens = tokenize(text)

    return [
        word2idx.get(
            token,
            word2idx["<UNK>"]
        )
        for token in tokens
    ]
print(encode("this is cool"))

In [ ]:
MAX_LEN = 100

def pad_sequence(seq):

    if len(seq) > MAX_LEN:
        seq = seq[:MAX_LEN]

    seq = seq + [0]*(MAX_LEN-len(seq))

    return seq

In [ ]:
class ToxicDataset(Dataset):

    def __init__(
        self,
        texts,
        labels
    ):

        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):

        return len(self.texts)

    def __getitem__(
        self,
        idx
    ):

        encoded = encode(
            self.texts[idx]
        )

        padded = pad_sequence(
            encoded
        )

        return (
            torch.tensor(
                padded,
                dtype=torch.long
            ),
            torch.tensor(
                self.labels[idx],
                dtype=torch.float32
            )
        )

In [ ]:
train_dataset = ToxicDataset(
    train_texts,
    train_labels
)

val_dataset = ToxicDataset(
    val_texts,
    val_labels
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

In [ ]:
class ToxicLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self,x):

        embedded = self.embedding(x)

        output,(hidden,cell) = self.lstm(
            embedded
        )
        hidden_cat = torch.cat((hidden[0], hidden[1]), dim=1)
        logits = self.fc(
            hidden_cat
        )

        return logits.squeeze(1)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = ToxicLSTM(
    vocab_size=len(word2idx)
).to(device)

In [ ]:
pos_weight = torch.tensor([3.0]).to(device)



criterion = nn.BCEWithLogitsLoss(

    pos_weight=pos_weight

)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
sample_batch = next(
    iter(train_loader)
)

inputs,labels = sample_batch

inputs = inputs.to(device)

outputs = model(inputs)

print(outputs.shape)

In [ ]:
print(inputs.shape)
print(outputs.shape)

In [ ]:
EPOCHS = 8

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for inputs,labels in tqdm(train_loader):

        inputs = inputs.to(device)

        labels = labels.to(
            device,
            dtype=torch.float32
        )

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} | Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

# --- Threshold tuning ---
precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = f1_scores.argmax()
best_threshold_lstm = thresholds[best_idx]
print(f"Best threshold: {best_threshold_lstm:.3f}  (F1={f1_scores[best_idx]:.4f})")

# --- Final predictions using best threshold ---
all_preds = (np.array(all_probs) > best_threshold_lstm).astype(int)

print(classification_report(all_labels, all_preds))
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")

# Save metrics for final comparison
lstm_p1_auc = roc_auc_score(all_labels, all_probs)
lstm_p1_f1 = f1_scores[best_idx]

In [ ]:
auc = roc_auc_score(
    all_labels,
    all_probs
)

print(f"ROC-AUC Score: {auc:.4f}")

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_preds
)

print(cm)

lstm_p1_tn, lstm_p1_fp, lstm_p1_fn, lstm_p1_tp = cm.ravel()

## 3. LSTM — Phase 2: Train on Full `train.csv`, Evaluate on Official Test

In [ ]:
#training again on the full dataset
full_dataset = ToxicDataset(
    df["comment_text"],
    df["label"]
)

full_loader = DataLoader(
    full_dataset,
    batch_size=64,
    shuffle=True
)

model = ToxicLSTM(
    vocab_size=len(word2idx)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)


sample_batch = next(
    iter(full_loader)
)

inputs,labels = sample_batch

inputs = inputs.to(device)

outputs = model(inputs)

print(outputs.shape)

In [ ]:
EPOCHS = 8

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for inputs,labels in tqdm(full_loader):

        inputs = inputs.to(device)

        labels = labels.to(
            device,
            dtype=torch.float32
        )

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(full_loader)

    print(
        f"Epoch {epoch+1} | Loss: {avg_loss:.4f}"
    )

In [ ]:
# Evaluate on OFFICIAL test set using threshold from Phase 1

official_test_dataset = ToxicDataset(
    test_df["comment_text"],
    test_df["label"]
)

official_test_loader = DataLoader(
    official_test_dataset,
    batch_size=64
)

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():

    for inputs, labels in official_test_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        probs = torch.sigmoid(outputs)

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ROC-AUC (threshold independent)

auc = roc_auc_score(
    all_labels,
    all_probs
)

print(
    f"ROC-AUC Score: {auc:.4f}"
)

# Find best threshold using F1

precision, recall, thresholds = precision_recall_curve(
    all_labels,
    all_probs
)

f1_scores = (
    2 * precision * recall
) / (
    precision + recall + 1e-8
)

best_idx = np.argmax(
    f1_scores[:-1]
)

best_threshold = thresholds[
    best_idx
]

best_f1 = f1_scores[
    best_idx
]

print(
    f"Best Threshold: {best_threshold:.4f}"
)

print(
    f"Best F1 Score: {best_f1:.4f}"
)

# Predictions using threshold from Phase 1

all_preds = (
    all_probs > best_threshold_lstm
).astype(int)

print("\n==========================")
print("LSTM OFFICIAL TEST RESULTS")
print("==========================\n")

print(
    classification_report(
        all_labels,
        all_preds
    )
)

cm = confusion_matrix(
    all_labels,
    all_preds
)

print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print(
    f"\nTN={tn} FP={fp} FN={fn} TP={tp}"
)

print(f"Best Threshold Used: {best_threshold_lstm:.4f}")
print(f"\nOfficial Test Samples: {len(test_df)}")

# Save metrics for final comparison
lstm_p2_auc = auc
lstm_p2_tn, lstm_p2_fp, lstm_p2_fn, lstm_p2_tp = tn, fp, fn, tp

In [ ]:
torch.save(model.state_dict(), "toxic_lstm.pt")
print("LSTM model saved.")

## 4. TF-IDF + MLP — Phase 1: Train on Split, Evaluate on Validation

In [ ]:
# Fit TF-IDF on train split ONLY (no data leakage)

tfidf_vectorizer_split = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    strip_accents="unicode",
    analyzer="word",
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer_split.fit_transform(
    train_texts
)

X_val_tfidf = tfidf_vectorizer_split.transform(
    val_texts
)

print(f"TF-IDF train shape: {X_train_tfidf.shape}")
print(f"TF-IDF val shape: {X_val_tfidf.shape}")

In [ ]:
# Undersample majority class gently — 90/10 -> 70/30
rus = RandomUnderSampler(
    sampling_strategy=0.3,
    random_state=42
)

X_train_bal, y_train_bal = rus.fit_resample(
    X_train_tfidf,
    train_labels
)

print(f"After undersampling — Class dist: {np.bincount(y_train_bal)}")
print("Training on split (for threshold tuning)...")

threshold_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=20,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=3,
    batch_size=512,
    random_state=42,
    verbose=True
)

threshold_model.fit(X_train_bal, y_train_bal)

In [ ]:
# -----------------------------------
# FIND BEST THRESHOLD ON VAL SET
# -----------------------------------

val_probs = threshold_model.predict_proba(X_val_tfidf)[:, 1]

precision_val, recall_val, thresholds_val = precision_recall_curve(
    val_labels,
    val_probs
)

f1_val = (
    2 * precision_val * recall_val
) / (
    precision_val + recall_val + 1e-8
)

min_precision = 0.60
valid_mask = precision_val[:-1] >= min_precision

if valid_mask.any():
    filtered_f1 = np.where(valid_mask, f1_val[:-1], 0)
    best_idx = np.argmax(filtered_f1)
else:
    best_idx = np.argmax(f1_val[:-1])

best_threshold_tfidf = thresholds_val[best_idx]

print(f"\nBest Threshold (from val set): {best_threshold_tfidf:.4f}")
print(f"Best Val F1: {f1_val[best_idx]:.4f}")

# Evaluate on val set with threshold
val_preds = (val_probs > best_threshold_tfidf).astype(int)

print(classification_report(val_labels, val_preds))
print(f"Val ROC-AUC: {roc_auc_score(val_labels, val_probs):.4f}")

cm = confusion_matrix(
    val_labels,
    val_preds
)

print(cm)

# Save metrics for final comparison
tfidf_p1_auc = roc_auc_score(val_labels, val_probs)
tfidf_p1_tn, tfidf_p1_fp, tfidf_p1_fn, tfidf_p1_tp = cm.ravel()

## 5. TF-IDF + MLP — Phase 2: Train on Full `train.csv`, Evaluate on Official Test

In [ ]:
# Refit TF-IDF on full training data

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    strip_accents="unicode",
    analyzer="word",
    stop_words="english"
)

tfidf_X_full = tfidf_vectorizer.fit_transform(
    df["comment_text"]
)

full_labels = df["label"]

print(f"TF-IDF full shape: {tfidf_X_full.shape}")

In [ ]:
# Undersample full training data
rus_full = RandomUnderSampler(
    sampling_strategy=0.3,
    random_state=42
)

X_full_bal, y_full_bal = rus_full.fit_resample(
    tfidf_X_full,
    full_labels
)

print(f"\nFull data after undersampling — Class dist: {np.bincount(y_full_bal)}")
print("Retraining on FULL dataset...")

tfidf_mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=20,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=3,
    batch_size=512,
    random_state=42,
    verbose=True
)

tfidf_mlp_model.fit(X_full_bal, y_full_bal)

In [ ]:
# -----------------------------------
# EVALUATE ON OFFICIAL TEST SET
# -----------------------------------

tfidf_X_test = tfidf_vectorizer.transform(
    test_df["comment_text"]
)

official_test_labels = test_df["label"].values

tfidf_test_probs = (
    tfidf_mlp_model.predict_proba(tfidf_X_test)[:, 1]
)

test_auc = roc_auc_score(
    official_test_labels,
    tfidf_test_probs
)

print(f"\nOfficial Test ROC-AUC: {test_auc:.4f}")

tfidf_test_preds = (
    tfidf_test_probs > best_threshold_tfidf
).astype(int)

# -----------------------------------
# FINAL REPORTS
# -----------------------------------

print("\n==========================")
print("TF-IDF OFFICIAL TEST RESULTS")
print("==========================\n")

print(
    classification_report(
        official_test_labels,
        tfidf_test_preds
    )
)

cm = confusion_matrix(
    official_test_labels,
    tfidf_test_preds
)

print("\nConfusion Matrix:\n")
print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Best Threshold Used: {best_threshold_tfidf:.4f}")
print(f"\nOfficial Test Samples: {len(test_df)}")

# Save metrics for final comparison
tfidf_p2_auc = test_auc
tfidf_p2_tn, tfidf_p2_fp, tfidf_p2_fn, tfidf_p2_tp = tn, fp, fn, tp

In [ ]:
import joblib

joblib.dump(tfidf_mlp_model, "toxic_tfidf_mlp.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")
print("TF-IDF model and vectorizer saved.")

## Results Summary

In [ ]:
# =============================================
# COMPREHENSIVE MODEL COMPARISON
# =============================================

def compute_metrics(tn, fp, fn, tp, auc, threshold):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {
        'Threshold': f'{threshold:.4f}',
        'Accuracy': f'{accuracy:.4f}',
        'Precision (Toxic)': f'{precision:.4f}',
        'Recall (Toxic)': f'{recall:.4f}',
        'F1-Score (Toxic)': f'{f1:.4f}',
        'ROC-AUC': f'{auc:.4f}',
        'True Positives': tp,
        'False Positives': fp,
        'True Negatives': tn,
        'False Negatives': fn,
        'Total Samples': tp + fp + tn + fn
    }

results = pd.DataFrame({
    'LSTM (Val Split)': compute_metrics(
        lstm_p1_tn, lstm_p1_fp, lstm_p1_fn, lstm_p1_tp,
        lstm_p1_auc, best_threshold_lstm
    ),
    'LSTM (Official Test)': compute_metrics(
        lstm_p2_tn, lstm_p2_fp, lstm_p2_fn, lstm_p2_tp,
        lstm_p2_auc, best_threshold_lstm
    ),
    'TF-IDF (Val Split)': compute_metrics(
        tfidf_p1_tn, tfidf_p1_fp, tfidf_p1_fn, tfidf_p1_tp,
        tfidf_p1_auc, best_threshold_tfidf
    ),
    'TF-IDF (Official Test)': compute_metrics(
        tfidf_p2_tn, tfidf_p2_fp, tfidf_p2_fn, tfidf_p2_tp,
        tfidf_p2_auc, best_threshold_tfidf
    ),
})

print("=" * 80)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 80)
print()
print(results.to_string())
print()
print("=" * 80)


# =============================================
# 6. Error Analysis: Examining Misclassified Examples
# =============================================

To fully understand the model failures and analyze the precision-recall trade-offs, we must examine the actual text of the comments that our models misclassified (False Positives and False Negatives).


In [ ]:
import pandas as pd

# We will look at the predictions from the official test set (Phase 2)
# Let's extract the raw texts and the true labels
test_texts = test_df["comment_text"].values
true_labels = official_test_labels

# Identify LSTM failures
lstm_fp_idx = np.where((all_preds == 1) & (true_labels == 0))[0]
lstm_fn_idx = np.where((all_preds == 0) & (true_labels == 1))[0]

# Identify TF-IDF failures
tfidf_fp_idx = np.where((tfidf_test_preds == 1) & (true_labels == 0))[0]
tfidf_fn_idx = np.where((tfidf_test_preds == 0) & (true_labels == 1))[0]

def print_examples(indices, texts, title, num_examples=3):
    print(f"--- {title} ---")
    for i in indices[:num_examples]:
        print(f"- {texts[i][:200]}...")
    print("\n")

print("================ LSTM FAILURES ================")
print_examples(lstm_fp_idx, test_texts, "LSTM False Positives (Clean comments predicted as Toxic)")
print_examples(lstm_fn_idx, test_texts, "LSTM False Negatives (Toxic comments predicted as Clean)")

print("================ TF-IDF FAILURES ================")
print_examples(tfidf_fp_idx, test_texts, "TF-IDF False Positives (Clean comments predicted as Toxic)")
print_examples(tfidf_fn_idx, test_texts, "TF-IDF False Negatives (Toxic comments predicted as Clean)")


## Analysis of Misclassifications

**1. False Positives (Predicted Toxic, Actually Clean)**
- Often, these occur because the comment contains words that are highly correlated with toxicity in the training set, but are being used in a benign, clinical, or quoting context (e.g., discussing a Wikipedia article about a controversial topic).
- The TF-IDF model is especially vulnerable to this because it looks at words independently (or as bigrams) and lacks deep contextual understanding.

**2. False Negatives (Predicted Clean, Actually Toxic)**
- These often occur when users employ sarcasm, subtle threats, or obfuscated profanity (e.g., replacing letters with symbols like `f*ck`).
- The LSTM struggles with out-of-vocabulary (OOV) words if the obfuscated words weren't seen during training. 
- TF-IDF also struggles if the specific toxic n-gram wasn't frequent enough in the training data.

**Precision-Recall Trade-off:**
By adjusting the classification threshold (as we did in Phase 1 using the `precision_recall_curve`), we balance these two errors. Lowering the threshold increases Recall (catching more toxic comments) but hurts Precision (flagging more clean comments as toxic), resulting in more False Positives.